# Exercise 4 — Combined Signal

`combined_signal` is the integration function. It requires both the SMA crossover (trend confirmation) AND the MACD cross (momentum confirmation) to be positive before going long. Requiring two independent signals to agree reduces false entries at the cost of entering later in the trend. This is the classical 'signal confluence' approach used in systematic strategies.

In [ ]:
import pandas as pd, math, warnings

def _synthetic(n=252):
    prices = [100.0 * (1 + 0.3 * math.sin(i * 2 * math.pi / n)) for i in range(n)]
    dates  = pd.date_range("2023-01-01", periods=n, freq="B")
    close  = pd.Series(prices, index=dates)
    return pd.DataFrame({
        "Open":   close.shift(1).fillna(close.iloc[0]),
        "High":   close * 1.01,
        "Low":    close * 0.99,
        "Close":  close,
        "Volume": pd.Series([1_000_000 + i * 1_000 for i in range(n)], index=dates),
    })

def _sma(s, w):  return s.rolling(w).mean()
def _ema(s, w):  return s.ewm(span=w, adjust=False).mean()
def _rsi(s, w):
    d = s.diff()
    g = d.clip(lower=0).rolling(w).mean()
    l = (-d.clip(upper=0)).rolling(w).mean()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        rs = g / l
    return 100 - (100 / (1 + rs))
def sma_crossover(df, fast=20, slow=50):
    close = df["Close"]
    return (_sma(close, fast) > _sma(close, slow)).fillna(False).astype(int)
def rsi_mean_reversion(df, window=14, oversold=30, overbought=70):
    rsi_s  = _rsi(df["Close"], window)
    signal = pd.Series(float("nan"), index=df.index)
    signal[rsi_s < oversold]   = 1.0
    signal[rsi_s > overbought] = 0.0
    return signal.ffill().fillna(0).astype(int)
def macd_cross(df, fast=12, slow=26, signal=9):
    close       = df["Close"]
    macd_line   = _ema(close, fast) - _ema(close, slow)
    signal_line = _ema(macd_line, signal)
    return (macd_line > signal_line).astype(int)

def combined_signal(df, fast=20, slow=50, macd_fast=12, macd_slow=26, macd_sig=9):
    """Long only when SMA crossover AND MACD cross both agree.

    Steps:
      1. sma_sig  = sma_crossover(df, fast, slow)
      2. macd_sig = macd_cross(df, macd_fast, macd_slow, macd_sig)
      3. return ((sma_sig == 1) & (macd_sig == 1)).astype(int)

    More conservative than either signal alone.
    """
    # TODO: implement the 3 steps above
    return pd.Series(0, index=df.index)


### Checks

In [ ]:
checks = 0

# 1 — returns same-length Series, no NaN, values in {0,1}
try:
    df  = _synthetic()
    sig = combined_signal(df)
    assert isinstance(sig, pd.Series) and len(sig) == len(df)
    assert not sig.isna().any()
    assert set(sig.unique()).issubset({0, 1})
    checks += 1; print("✅ 1 valid Series: same length, no NaN, values in {0,1}")
except Exception as e:
    print("❌ 1:", e)

# 2 — combined is more conservative: sum(combined) <= sum(sma_cross) and sum(macd_cross)
try:
    df       = _synthetic()
    sma_sig  = sma_crossover(df, 20, 50)
    macd_sig = macd_cross(df)
    comb_sig = combined_signal(df)
    assert comb_sig.sum() <= sma_sig.sum(),         f"combined ({comb_sig.sum()}) should have ≤ 1s than SMA ({sma_sig.sum()})"
    assert comb_sig.sum() <= macd_sig.sum(),         f"combined ({comb_sig.sum()}) should have ≤ 1s than MACD ({macd_sig.sum()})"
    checks += 1; print("✅ 2 combined is more conservative (fewer or equal 1s)")
except Exception as e:
    print("❌ 2:", e)

# 3 — combined = 1 only when both sma and macd are 1
try:
    df       = _synthetic()
    sma_sig  = sma_crossover(df, 20, 50)
    macd_sig = macd_cross(df)
    comb_sig = combined_signal(df)
    expected = ((sma_sig == 1) & (macd_sig == 1)).astype(int)
    assert (comb_sig == expected).all(), "combined != sma AND macd"
    checks += 1; print("✅ 3 combined = sma_cross AND macd_cross")
except Exception as e:
    print("❌ 3:", e)

# 4 — combined = 0 when either signal is 0
try:
    df       = _synthetic()
    sma_sig  = sma_crossover(df, 20, 50)
    macd_sig = macd_cross(df)
    comb_sig = combined_signal(df)
    # Wherever either is 0, combined must be 0
    either_zero = (sma_sig == 0) | (macd_sig == 0)
    assert (comb_sig[either_zero] == 0).all()
    checks += 1; print("✅ 4 combined = 0 whenever either component is 0")
except Exception as e:
    print("❌ 4:", e)

# 5 — combined = 1 only where both are 1
try:
    df       = _synthetic()
    sma_sig  = sma_crossover(df, 20, 50)
    macd_sig = macd_cross(df)
    comb_sig = combined_signal(df)
    both_one = (sma_sig == 1) & (macd_sig == 1)
    assert (comb_sig[both_one] == 1).all()
    checks += 1; print("✅ 5 combined = 1 only where both signals are 1")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
